In [9]:
import numpy as np

def elastic_1d(m1, m2, u1, u2):
    """Return post-collision velocities for a 1D elastic collision."""
    v1 = ((m1 - m2) * u1 + 2 * m2 * u2) / (m1 + m2)
    v2 = ((m2 - m1) * u2 + 2 * m1 * u1) / (m1 + m2)
    return v1, v2

# Classic check: equal masses exchange velocities
print(elastic_1d(1.0, 1.0, 3.0, 0.0))   # -> (0.0, 3.0)

(0.0, 3.0)


🔬 Three limiting cases worth memorizing: equal masses exchange velocities; a light object hitting a massive wall reverses (v → −u); a massive object hitting a light one barely slows. If your code reproduces all three, it's almost certainly correct.

In [10]:
def collide_1d(m1, m2, u1, u2, e=1.0):
    """1D collision with restitution e. e=1 elastic, e=0 perfectly inelastic."""
    total_m = m1 + m2
    # center-of-mass velocity is unchanged by the collision
    v_cm = (m1 * u1 + m2 * u2) / total_m
    v1 = v_cm - e * (m2 / total_m) * (u1 - u2)
    v2 = v_cm + e * (m1 / total_m) * (u1 - u2)
    return v1, v2

✍️ Try it: Call collide_1d with e=1, e=0.5, and e=0 for a 2 kg block at 4 m/s hitting a stationary 3 kg block. Compute total momentum and total KE before and after each. Confirm momentum is identical in all three, and KE decreases as e shrinks.

In [11]:
print(f'e = 1 "elastic":{collide_1d(2.0, 3.0, 4.0, 0.0, 1.0)}')
print(f'e = 0.5 "partially elastic":{collide_1d(2.0, 3.0, 4.0, 0.0, 0.5)}')
print(f'e = 0 "inelastic":{collide_1d(2.0, 3.0, 4.0, 0.0, 0.0)}')

e = 1 "elastic":(-0.7999999999999998, 3.2)
e = 0.5 "partially elastic":(0.40000000000000013, 2.4000000000000004)
e = 0 "inelastic":(1.6, 1.6)


💡 Key Idea: For frictionless circular bodies, the collision impulse acts only along the line connecting the two centers (the "normal" direction). The velocity components perpendicular to that line are unchanged. So a 2D collision is just a 1D collision along the line of centers, with the tangential components carried along untouched.

In [12]:
def collide_2d(m1, m2, r1, r2, v1, v2, e=1.0):
    """Resolve a 2D collision between two circles. Positions r, velocities v are 2-vectors."""
    r1, r2 = np.asarray(r1, float), np.asarray(r2, float)
    v1, v2 = np.asarray(v1, float), np.asarray(v2, float)

    n = r2 - r1
    dist = np.linalg.norm(n)
    if dist == 0:
        return v1, v2            # degenerate; skip
    n = n / dist                 # unit normal

    # normal (scalar) components along n
    u1n, u2n = np.dot(v1, n), np.dot(v2, n)
    # tangential vector components (unchanged)
    v1t = v1 - u1n * n
    v2t = v2 - u2n * n

    # 1D collision on the normal components
    new1n, new2n = collide_1d(m1, m2, u1n, u2n, e)

    return v1t + new1n * n, v2t + new2n * n

🔬 Why this works: A frictionless contact can only push, and it can only push along the normal. There's no mechanism to change the tangential motion. This decomposition is exactly what commercial 2D physics engines do — you're building the real thing.

In [13]:
def overlapping(r1, r2, radius1, radius2):
    return np.linalg.norm(np.asarray(r2) - np.asarray(r1)) < (radius1 + radius2)

⚠️ The "sticky ball" bug: The most common collision bug is objects that clump and vibrate. The cause is almost always detecting a new collision every step because you never separated the overlapping pair. Two standard fixes: (1) positional correction — push the bodies apart along the normal so they just touch; or (2) only resolve if they are approaching (relative normal velocity points inward). We'll use the approaching-check in Lab 4; it's one line and kills most of these bugs.

In [ ]:
# Only resolve if the bodies are actually moving toward each other
approaching = np.dot(v2 - v1, r2 - r1) < 0
if overlapping(r1, r2, R1, R2) and approaching:
    v1, v2 = collide_2d(m1, m2, r1, r2, v1, v2, e)

# Should probably be put into the definition of collide_2d

NameError: name 'R1' is not defined

In [16]:
def total_momentum(bodies):
    return sum(b['m'] * b['v'] for b in bodies)   # vector sum

def total_ke(bodies):
    return sum(0.5 * b['m'] * np.dot(b['v'], b['v']) for b in bodies)

💡 The habit that scales: Notice that total_momentum and total_ke loop over a list of bodies, each a dictionary with 'm', 'r', 'v'. That list-of-bodies pattern is the embryo of the World class you'll build in Week 7. Start thinking of your simulation as "a collection of bodies plus rules that act on them."